In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import cvxpy as cp
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

## $\text{\textcolor{green}{Preprocessing data}}$

In [ ]:
df_temp = pd.read_csv(ROOT / "data/complete_dataframe2_30min.csv",delimiter=",",decimal=".",parse_dates=["ts"],index_col="ts")
df_load = pd.read_csv(ROOT / "data/archive/archive/building_consumption.csv", parse_dates=["timestamp"])
df_price = pd.read_csv(ROOT / "data/DayAheadData_fulltimeperiod.csv")
df_price["ts"] = pd.to_datetime(df_price['ts'])
df_price = df_price.set_index("ts") # set time as index



filtered = df_load[
    (df_load["campus_id"] == 1) &
    (df_load["timestamp"] >= "2021-05-14") &
    (df_load["timestamp"] <= "2022-04-11")
]
# How many 15-min readings are there available per meter_id?
counts = filtered.groupby("meter_id").size().reset_index(name="num_readings")

In [ ]:
df_temp2 = df_temp.join(df_price,how="inner")
df_temp2 = df_temp2[:-1]

In [ ]:
downsizing_factor = 0.01
total_load = filtered.groupby("timestamp")["consumption"].sum().reset_index(name="total_consumption")
total_load["timestamp"] = pd.to_datetime(total_load["timestamp"])+pd.DateOffset(years=4)
total_load = total_load.set_index("timestamp")

total_load_30min = total_load.resample("30min").mean() * downsizing_factor

In [ ]:
df = df_temp2.join(total_load_30min,how="inner")

In [ ]:
# Confirm data has 48-step days
total_load_30min["date"] = total_load_30min.index.date
load_counts = total_load_30min.groupby("date").size()
load_complete_days = load_counts[load_counts == 48].index

load_days_df = total_load_30min[total_load_30min["date"].isin(load_complete_days)].copy()

# Make days into 48-step vectors

load_day_profiles = []

for _, day_df in load_days_df.groupby("date"):
    day_profile = day_df.sort_index()["total_consumption"].to_numpy()
    load_day_profiles.append(day_profile)

# Visualization of daily profiles <-- Would the MW scale be per time step or across the day profile?

daily_totals = total_load_30min.groupby(total_load_30min.index.date)["total_consumption"].sum()
daily_energy = daily_totals * 0.5

In [ ]:
# Aggregate df across PV modules and wind turbines
df_agg = df.copy()
df_agg["PV_total"] = df_agg[
    ["B117_m", "B319_m", "B330_1_m", "B330_2_m", "B330_3_m", "B716_m", "B715_m"]
].sum(axis=1, min_count=1)

df_agg["Wind_total"] = df_agg[
    ["Aircon_WT Power_m", "Gaia_WT Power_m"]
].sum(axis=1, min_count=1)

df_agg.dropna(subset=["PV_total", "Wind_total"], how="any", inplace=True)
df_agg = df_agg.drop(columns=["Aircon_WT Power_m", "Gaia_WT Power_m","B117_m", "B319_m", "B330_1_m", "B330_2_m",
                               "B330_3_m", "B716_m", "B715_m","wind_dir","wind_max","wind_min","wind_speed","radia_glob","temp_dry"])
# Keep only complete days
df_agg["date"] = df_agg.index.date
counts_per_day = df_agg.groupby("date").size()
complete_days = counts_per_day[counts_per_day == 48].index
df_agg = df_agg[df_agg["date"].isin(complete_days)].copy()

In [ ]:
df_agg

## $\text{\textcolor{green}{Generation vs Load}}$

In [ ]:
df_grouped = {str(date): group for date, group in df_agg.groupby("date")}

In [ ]:
## PLOT DAILY PROFILE OF LOAD AND POWER 
days = ["2025-07-15","2025-10-15","2026-01-15","2026-04-09"]
cols = ["PV_total","Wind_total","total_consumption"]
# Calculate number of subplots needed
n_days = len(days)
n_cols = 1
n_rows = 4

# Create subplots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(10, 10))
axes = axes.flatten()  # Flatten to easily index

# Plot each day
for i, day in enumerate(days):
    df_day = df_grouped[day]  
    
    # Plot different parameters (columns) vs time
    # Assuming 'timestamp' or 'time' is your x-axis column
    for column in cols:
        axes[i].plot(df_day.index.strftime('%H:%M'), df_day[column], label=column, marker='o')
    
    axes[i].set_title(f'Day: {day}')
    axes[i].set_xlabel('Time')
    axes[i].set_ylabel('kW')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)
    axes[i].tick_params(axis='x', rotation=45)

# Remove any unused subplots
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

## $\text{\textcolor{green}{Implementing the solver with "Perfect Knowledge" }}$

In [ ]:
# Daily solver
def solve_day_profile(df, minimum_self_sufficiency, E_capacity, P_capacity, charge_eff, discharge_eff, dT, C_rate):
    PV_gen = df["PV_total"].to_numpy()
    Wind_gen = df["Wind_total"].to_numpy()
    n = len(df)
    load = df["total_consumption"].to_numpy()
    price = df["DayAheadPriceDKK"].to_numpy()
    
    if P_capacity > C_rate * E_capacity:
        print(f"Warning: P_capacity={P_capacity}kW exceeds C_rate * E_capacity = {C_rate * E_capacity}")

    # Constants
    Pload = cp.Constant(load)
    Pgen_PV = cp.Constant(PV_gen)
    Pgen_W = cp.Constant(Wind_gen)
    P_capacity = cp.Constant(P_capacity)
    E_capacity = cp.Constant(E_capacity)

    # Variables

    Pcharge = cp.Variable(n, nonneg=True)
    Pdischarge = cp.Variable(n, nonneg=True)

    Pgrid_buy = cp.Variable(n) # if positive, energy is being bought from the grid. if negative we are selling energy to the grid. 

    SOC = cp.Variable(n)

    constraints = [
        Pcharge <= P_capacity, #### should it include the mask z or not? 
        Pdischarge <= P_capacity,

        SOC >= 0.1 * E_capacity,
        SOC <= E_capacity,

        # Power balance 
        Pgen_PV + Pgen_W + Pdischarge + Pgrid_buy == Pload + Pcharge,

        SOC[0] == 0.5 * E_capacity + (Pcharge[0] * charge_eff - Pdischarge[0] / discharge_eff) * dT,
        SOC[1:] == SOC[:-1] + (Pcharge[1:] * charge_eff - Pdischarge[1:] / discharge_eff) * dT,

        SOC[n - 1] >= 0.5 * E_capacity # Do we want this?  
        ]


    cost_energy_buy = cp.sum(cp.multiply(Pgrid_buy, price)) * dT 
    #DKK 
    

    problem = cp.Problem(cp.Minimize(cost_energy_buy), constraints)
    
    try:
        problem.solve(solver=cp.OSQP, verbose=False,max_iter=1000000)



        if problem.status not in ["optimal", "optimal_inaccurate"]:
            print(f"Problem arrose: {problem.status}")
            return {
            "status": problem.status,
            "Total_cost_DKK": np.nan,
            "Cost_of_energy_DKK": np.nan,
            "bought_energy_kWh": np.nan,
            "sold_energy_kWh": np.nan,

            "Pgrid_buy": np.nan,
            "Pgrid_sell": np.nan,
            "Pcharge": np.nan,
            "Pdischarge": np.nan,
            "SOC": np.nan
            }
        
        Pgrid_sell = cp.neg(Pgrid_buy)
        Pgrid_buy = cp.pos(Pgrid_buy)

        return {
        
            "status": problem.status,
            "Total_cost_DKK": (cost_energy_buy).value,
            "Cost_of_bought_energy_DKK": cp.sum(cp.multiply(Pgrid_buy, price)) * dT,
            "bought_energy_kWh": np.sum(Pgrid_buy.value) * dT,
            "sold_energy_kWh": np.sum(Pgrid_sell.value) * dT,

            "Pgrid_buy": Pgrid_buy.value,
            "Pgrid_sell": Pgrid_sell.value,
            "Pcharge": Pcharge.value,
            "Pdischarge": Pdischarge.value,
            "SOC": SOC.value
        }

    except Exception as e:
        print(f"Error arose {type(e).__name__}")
        return {
            "status": type(e).__name__,
            "Total_cost_DKK": np.nan,
            "Cost_of_energy_DKK": np.nan,
            "bought_energy_kWh": np.nan,
            "sold_energy_kWh": np.nan,

            "Pgrid_buy": np.nan,
            "Pgrid_sell": np.nan,
            "Pcharge": np.nan,
            "Pdischarge": np.nan,
            "SOC": np.nan
        }

## $\text{\textcolor{green}{Running Solver}}$

In [ ]:
# Settings
dT = 0.5
charge_eff = 0.95
discharge_eff = 0.95
C_rate = 1
P_capacity = 10 ###### ADD RESULT FROM EMIs OPTIMIZATION 
E_capacity = 10 ####### ADD RESULT FROM EMIs OPTIMIZATION 

min_self_sufficiency = 0 # NO REQUIREMENT

In [ ]:
# Run the Optimization for the given days
profiles = []

days = ["2025-07-15","2025-10-15","2026-01-15","2026-04-09"]

for day in days:
    print("---------------------------------------------------------")
    print("Optimization starting for ", day)
    print("---------------------------------------------------------")
    day_df = df_grouped[day]

    out = solve_day_profile(
        day_df,
        min_self_sufficiency,
        E_capacity,
        P_capacity,
        charge_eff,
        discharge_eff,
        dT,
        C_rate
    )

    # Build time-series dataframe
    profile_df = pd.DataFrame({
        "ts": day_df.index,
        "P_buy": out["Pgrid_buy"],
        "P_sell": out["Pgrid_sell"],
        "P_charge": out["Pcharge"],
        "P_discharge": out["Pdischarge"],

        "SOC": out["SOC"],
        "price": day_df["DayAheadPriceDKK"].values,
        "load": day_df["total_consumption"].values,
        "PV": day_df["PV_total"].values,
        "Wind": day_df["Wind_total"].values,
    })

    profile_df["date"] = day
    print(f"Status for the optimization for {day}: {out["status"]} ")
    print(f"Total cost is: {out["Total_cost_DKK"]}")
    print(f"Amount of bought energy is: {out["bought_energy_kWh"]}")
    print(f"Amount of sold energy is: {out["sold_energy_kWh"]}")

    profiles.append(profile_df)


profiles_df = pd.concat(profiles, ignore_index=True)
profiles_df = profiles_df.set_index("ts")

## $\text{\textcolor{green}{Analyzing Solver Output}}$

In [ ]:
# Plot the Daily Power Profiles
import matplotlib.dates as mdates
days = ["2025-07-15","2025-10-15","2026-01-15","2026-04-09"]

pos_cols = ["PV",	"Wind", "P_discharge", "P_buy"]
pos_color = ["gold","lightskyblue","plum","lightgreen"]
labels = ["PV", "Wind", "Battery", "Grid"]

neg_cols = ["P_charge","P_sell"]
neg_color = ["plum","lightgreen"]

n_days = len(days)
n_cols = 2
n_rows = 2

fig, axes = plt.subplots(n_rows, n_cols, figsize=(10, 9))

axes = axes.flatten()  # Flatten to easily index


for j, day in enumerate(days):

    day_plot = profiles_df[profiles_df["date"] == day]


    width = 0.85/len(day_plot) # the width of the bars: can also be len(x) sequence


    
    top = np.zeros(len(day_plot))
    bottom = np.zeros(len(day_plot))
    for i,col in enumerate(pos_cols): # Plotting all "positive" power
        color = pos_color[i]
        axes[j].bar(day_plot.index, day_plot[col], width, label=labels[i], bottom=top, color = color)

        top += day_plot[col]

    for i,col in enumerate(neg_cols): # Plotting all "negative" power
        color = neg_color[i]
        axes[j].bar(day_plot.index, - day_plot[col], width, bottom=bottom, color=color)

        bottom -= day_plot[col]

    axes[j].scatter(day_plot.index,day_plot["load"],label="Load",marker="o",color="lightsalmon") # Plot Load
    axes[j].set_xlabel("Time (hour)")
    axes[j].set_ylabel("Power (kW)")
    axes[j].set_title(day)
    
    axes[j].xaxis.set_major_locator(mdates.HourLocator(interval=2))
    axes[j].xaxis.set_major_formatter(mdates.DateFormatter('%H'))


handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.02), ncol=len(handles))
fig.tight_layout()
fig.show()